# PaperEngine on Google Colab

This notebook sets up **PaperEngine** — a local-first scholarly research acquisition
engine — on a Colab runtime, builds it from source at a pinned revision, and runs a
**bounded, real** discovery + open-access PDF acquisition workflow against live
scholarly sources.

**Important Colab facts:**
- The runtime's local disk is wiped when the session ends. Durable results must be
  copied to **Google Drive** (mounted below) before shutdown.
- Nothing here is required for normal local use — Colab is an optional execution
  environment. The engine itself has no Colab dependency.
- All requests are polite and bounded: tiny page budgets, rate-limited adapters,
  at most a couple of PDF downloads by default.

## 1. Install the Rust toolchain

PaperEngine is a Rust crate. We install a minimal stable toolchain (clippy + rustfmt
included). This takes one to two minutes.

In [ ]:
%%bash -e
if ! command -v cargo >/dev/null; then
  curl --proto '=https' --tlsv1.2 --retry 5 --retry-connrefused \
       --location --silent --show-error --fail \
       https://sh.rustup.rs | sh -s -- -y --profile minimal --component clippy,rustfmt
fi
source "$HOME/.cargo/env"
rustc --version && cargo --version

## 2. Obtain the repository at an identifiable revision

The notebook pins an exact commit SHA. Change `PINNED_REVISION` to `main` (or any
commit) as needed — the revision actually built is printed and recorded in the run
report provenance.

In [ ]:
%%bash -e
source "$HOME/.cargo/env"
PINNED_REVISION="82362f4e4b349881118dd5b7176b621a8f86b33c"  # verified 2026-10-19
if [ ! -d paperengine ]; then
  git clone https://github.com/Vtheonly/paperengine.git
fi
cd paperengine
git fetch origin main
git checkout "$PINNED_REVISION"
echo "BUILD_REVISION=$(git rev-parse HEAD)"
echo "BUILD_COMMIT_SUBJECT=$(git log -1 --pretty=%s)"

## 3. Build and verify

Release build plus the **offline** test suite. Offline tests never touch the network;
they run against recorded fixtures and a local HTTP server.

In [ ]:
%%bash -e
source "$HOME/.cargo/env"
cd paperengine
cargo build --release
cargo fmt --all -- --check
cargo clippy --all-targets --all-features -- -D warnings
cargo test --all-features

## 4. Configuration and source policy

Set a contact e-mail (used for polite API pools on OpenAlex/Crossref and required by
Unpaywall). The budgets below bound the whole run: at most 5 records discovered, at
most 2 PDFs downloaded, at most 64 MiB per file.

In [ ]:
%%bash -e
cd paperengine
cat > paperengine.toml <<'EOF'
schema_version = 1
contact_email = "paperengine@users.noreply.github.com"

[discovery]
default_page_size = 5
max_pages = 1
max_records = 5

[acquisition]
max_downloads = 2
max_bytes_per_file = 67108864
max_total_bytes = 134217728
EOF
./target/release/paperengine self-test

## 5. Bounded live discovery (metadata only — no downloads)

Discovery is structurally download-free: it writes normalized metadata records as
JSONL, including any open-access PDF URL the source itself reports.

In [ ]:
%%bash -e
cd paperengine
./target/release/paperengine discover \
    --source arxiv \
    --query "attention is all you need" \
    --max-records 5 \
    --out discovery.jsonl
echo "---- first record ----"
head -n 1 discovery.jsonl | python3 -m json.tool | head -30

### Optional: resolve an OA location through Unpaywall

Crossref-discovered works carry no direct PDF; Unpaywall resolves where an open-access
copy lives for a given DOI.

In [ ]:
%%bash -e
cd paperengine
./target/release/paperengine lookup --source unpaywall \
    --doi 10.1371/journal.pone.0000219

## 6. Real acquisition: download permitted OA PDFs

This is the production pipeline: policy re-check immediately before each request,
bounded resumable download, `%PDF-` signature gate, structural validation with lopdf,
quarantine of invalid payloads, and content-addressed storage with provenance
manifests and a run report.

In [ ]:
%%bash -e
cd paperengine
./target/release/paperengine acquire \
    --from discovery.jsonl \
    --library ./research-library \
    --max-downloads 2

## 7. Inspect the downloaded files and the execution report

In [ ]:
%%bash -e
cd paperengine
./target/release/paperengine inspect --library ./research-library
echo "---- latest run report ----"
latest_report=$(ls -1t research-library/reports/*.jsonl | head -n 1)
echo "$latest_report"
cat "$latest_report"

## 8. Persist results to Google Drive

Colab's local disk does not survive a session restart. Mount Drive and copy the whole
library — objects, provenance manifests, quarantine records, and reports — to a
durable folder.

> Adjust `DRIVE_DIR` if you prefer a different location. Everything the engine needs
> to resume later (hashes, provenance, budgets) is inside the library directory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os, datetime
DRIVE_DIR = '/content/drive/MyDrive/paperengine-library'
os.makedirs(DRIVE_DIR, exist_ok=True)
stamp = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
dest = os.path.join(DRIVE_DIR, f'run-{stamp}')
shutil.copytree('paperengine/research-library', dest)
print('saved to', dest)
for root, dirs, files in os.walk(dest):
    for name in files:
        path = os.path.join(root, name)
        print(f'{os.path.getsize(path):>10}  {path}')

## 9. Resume or inspect a completed run later

Re-running `acquire` with the same `--library` is safe: objects are content-addressed
and never clobbered, and byte-identical downloads deduplicate instead of duplicating.
Interrupted transfers leave at most a scratch file under `tmp/`, which the next run
clears before retrying.

```bash
# in a later session, after re-mounting Drive and re-cloning the repo:
cp -r "$DRIVE_DIR/run-<stamp>/." ./research-library
./target/release/paperengine acquire --from more-discovery.jsonl --library ./research-library
./target/release/paperengine inspect --library ./research-library
```

## What was verified where

- Offline checks (fmt / clippy / tests) — deterministic, no network.
- Live discovery + acquisition — real requests to arXiv and Unpaywall under strict
  budgets, recorded in the run report with byte sizes, SHA-256 hashes, page counts,
  and storage paths.
- OpenAlex search may report HTTP 429 on shared free-tier budgets; that is a stop
  condition the engine records honestly (configure `openalex_api_key` to lift it).

See `docs/SOURCE_COVERAGE.md` in the repository for the per-source status registry and
`docs/VERIFICATION.md` for the recorded evidence.